# Lab 3.2 — Authentication and Security Migration

**Duration:** ~7 min · **Level:** L300 · **Lab 3 of 4 — part 2/4**

Migrating the auth layer is where most teams get surprised. Key changes:

1. **IAM service prefix flips from `bedrock` to `bedrock-mantle`.**
   `AmazonBedrockFullAccess` does **not** grant Mantle access.
2. **Bearer tokens are added alongside SigV4.** They don't replace SigV4 —
   they sit next to it, with their own IAM, rotation, and CloudTrail story.
3. **Project ARNs** become a first-class IAM resource.
4. **Guardrails** are no longer a native request field — you call
   `ApplyGuardrail` separately or use `extra_headers`.

This notebook walks you through each change with runnable snippets.


In [1]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "src" / "common").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

import boto3
sts = boto3.client("sts")
iam = boto3.client("iam")
identity = sts.get_caller_identity()
print("caller:", identity["Arn"])


caller: arn:aws:iam::976939723775:user/mltest


## 1. IAM policy — Before

A typical runtime policy:

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Action": [
      "bedrock:InvokeModel",
      "bedrock:InvokeModelWithResponseStream",
      "bedrock:Converse",
      "bedrock:ConverseStream"
    ],
    "Resource": "arn:aws:bedrock:us-east-1::foundation-model/*"
  }]
}
```

Every `Action` is under the `bedrock:` prefix. Resource ARNs target
`foundation-model/*`.


## 1b. IAM policy — After (Mantle)

The Mantle equivalent. Note the service prefix (`bedrock-mantle`), the
new resource shape (`project/*`), and `CallWithBearerToken` — a permission-only
action that has to be on `Resource: "*"`.

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "MantleInference",
      "Effect": "Allow",
      "Action": [
        "bedrock-mantle:CreateInference",
        "bedrock-mantle:GetInference",
        "bedrock-mantle:CancelInference",
        "bedrock-mantle:GetModel",
        "bedrock-mantle:ListModels",
        "bedrock-mantle:GetProject",
        "bedrock-mantle:ListProjects"
      ],
      "Resource": "arn:aws:bedrock-mantle:*:*:project/*",
      "Condition": {
        "StringEquals": {
          "bedrock-mantle:Model": [
            "openai.gpt-oss-120b",
            "anthropic.claude-opus-4-7",
            "anthropic.claude-haiku-4-5"
          ]
        }
      }
    },
    {
      "Sid": "MantleCallWithBearerToken",
      "Effect": "Allow",
      "Action": "bedrock-mantle:CallWithBearerToken",
      "Resource": "*"
    }
  ]
}
```

The `bedrock-mantle:Model` condition key lets you list *exactly* which
models this role may invoke — a much finer grain than runtime's
`foundation-model/*` wildcard.


In [2]:
# Who am I? List the policies attached to my current principal so we can
# see whether we already have AmazonBedrockMantleInferenceAccess.
arn = identity["Arn"]

def my_role_name():
    # role-ARN shape: arn:aws:sts::<acct>:assumed-role/<role-name>/<session>
    if "assumed-role" in arn:
        return arn.split("/")[1]
    return None

rname = my_role_name()
if rname:
    try:
        attached = iam.list_attached_role_policies(RoleName=rname)["AttachedPolicies"]
        inline   = iam.list_role_policies(RoleName=rname)["PolicyNames"]
        print(f"role: {rname}")
        print("  attached:", [p["PolicyName"] for p in attached])
        print("  inline  :", inline)
    except Exception as e:
        print(f"(skipping IAM lookup: {e})")
else:
    print("not an assumed-role — skipping IAM lookup")


not an assumed-role — skipping IAM lookup


## 2. The three auth modes side-by-side

| Mode | Who issues | Lifetime | Typical workload | Pitfall |
|---|---|---|---|---|
| **SigV4 (runtime path)** | STS/IAM | Session (usually < 1h) | boto3 callers today | None — keep using it for runtime |
| **SigV4 (Mantle path)** | STS/IAM | Session | Go services, Lambdas that sign every call | SigV4 service name stays `bedrock`, **not** `bedrock-mantle` |
| **Short-term Bearer** | `aws-bedrock-token-generator` (from caller's IAM session) | ≤ 12 h | OpenAI SDK callers, notebooks | Region-scoped. Mint in the wrong region → 401 |
| **Long-term Bedrock API key** | `aws iam create-service-specific-credential --service-name bedrock.amazonaws.com` | Configurable (`--credential-age-days`) | CI, daemons | Creates a **dedicated IAM user** per key → governance sprawl |

The key is to **pick one per environment** and enforce it with an SCP.
Don't let one codebase mix short-term tokens, long-term keys, and SigV4 on
the same request path — your CloudTrail becomes impossible to reason about.


## 3. Cold-side test: can I actually hit Mantle?

This is the one-liner that surfaces most permission issues. If it prints
a non-empty list, you're good. If it raises `AccessDeniedException` on
`bedrock-mantle:ListModels`, re-read §1b.


In [3]:
from src.common.mantle import openai_client

mc = openai_client()
model_ids = [m.id for m in mc.models.list().data]
print(f"I can see {len(model_ids)} models.")
print("sample:", model_ids[:5])


I can see 40 models.
sample: ['zai.glm-5', 'moonshotai.kimi-k2.5', 'anthropic.claude-opus-4-7', 'openai.gpt-oss-safeguard-120b', 'nvidia.nemotron-nano-3-30b']


## 4. Guardrails — the hidden migration

On runtime, you attach a guardrail with a *first-class* request field:

```python
runtime.converse(
    modelId="...",
    messages=[...],
    guardrailConfig={"guardrailIdentifier": "..", "guardrailVersion": "1"},
)
```

On Mantle, there is **no `guardrailConfig` field**. You have three options:

1. **Out-of-band**: call `boto3.client("bedrock-runtime").apply_guardrail(…)`
   before / after your Mantle call. Two network round-trips per turn, but
   zero Mantle-side changes.
2. **`extra_body` / `extra_headers`**: some Mantle deployments expose
   guardrail IDs through provider-extension fields. Verify per model card.
3. **Wrap your client.** Put Mantle behind an adapter that owns the
   guardrail lifecycle. This is what most enterprises do in production.

The cell below demonstrates option 1.


In [4]:
# Illustrative. Supply a real guardrail ID to make this actually call AWS.
GUARDRAIL_ID = "REPLACE_ME"
TEXT = "Is drug X effective for disease Y? (user input)"

runtime_client = boto3.client("bedrock-runtime")
if GUARDRAIL_ID == "REPLACE_ME":
    print("(set GUARDRAIL_ID to a real guardrail in your account to run this cell.)")
else:
    gr = runtime_client.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion="DRAFT",
        source="INPUT",
        content=[{"text": {"text": TEXT}}],
    )
    if gr["action"] == "GUARDRAIL_INTERVENED":
        print("blocked by guardrail:", gr["outputs"][0]["text"]["text"])
    else:
        print("allowed — call Mantle here")


(set GUARDRAIL_ID to a real guardrail in your account to run this cell.)


## 5. CloudTrail

Mantle emits its own events. A few things to remember:

- Event source: `bedrock-mantle.amazonaws.com`.
- **Add a new Event Selector** to your existing trails — they don't
  auto-extend to the new service.
- Every Bearer-token call logs the originating principal under
  `userIdentity.arn` (short-term) or the dedicated IAM user (long-term).
- If a Bearer-token call 401s on the **client** side, you won't see a
  CloudTrail event at all (the request didn't make it to the service).
- `CallWithBearerToken` is what you'll most commonly see `AccessDenied` on
  during rollouts. Add a CloudWatch metric filter for it.

Add this metric filter to flag noisy auth issues:

```bash
aws logs put-metric-filter \
  --log-group-name /aws/cloudtrail/your-trail \
  --filter-name MantleBearerAccessDenied \
  --filter-pattern '{ $.eventName = "CallWithBearerToken" && $.errorCode = "AccessDenied*" }' \
  --metric-transformations \
      metricName=MantleBearerAccessDenied,metricNamespace=YourOrg/Mantle,metricValue=1
```


## 6. Migration checklist

- [ ] Decide policy strategy: AWS-managed (`AmazonBedrockMantleInferenceAccess`)
      vs custom inline. Default to managed for app roles, custom for admin.
- [ ] Decide key strategy: short-term Bearer in prod, long-term key in CI.
- [ ] Store long-term keys in Secrets Manager; rotate via Lambda.
- [ ] Add CloudTrail event selectors for `bedrock-mantle` events.
- [ ] Replace `guardrailConfig` with out-of-band `ApplyGuardrail` (or a
      wrapper).
- [ ] Add SCPs to restrict which accounts / regions can use Mantle.
- [ ] Verify VPC endpoint availability
      (`com.amazonaws.{region}.bedrock-mantle` — **verify** service name
      against latest docs per your region).
